# Обучение в Google Colab

Ноутбук только запускает скрипты из репозитория: вся логика лежит в `src/` и `scripts/`.

1. `Среда выполнения -> Сменить среду выполнения -> T4 GPU`
2. Репозиторий уже прописан в первой ячейке (`REPO_URL`).
3. Запускать ячейки по порядку.

**Если Colab отключился**, нужно перезапустить ячейки 1 и 3: завершённые эксперименты пропустятся,
а прерванный продолжится с последней эпохи (чекпоинты лежат на Google Drive).
Вкладку держите открытой: бесплатный Colab отключает неактивные сессии.

In [ ]:
# Общая подготовка: Google Drive, репозиторий, данные.
# В Colab: Среда выполнения -> Сменить среду выполнения -> T4 GPU.
import os, sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/semenovfilipp/image-classification.git"
BRANCH = "feature/image-classification"  # после merge в main поменять на "main"

REPO_DIR = "/content/image-classification" if IN_COLAB else os.getcwd()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/oxford_pets"   # данные и результаты живут здесь
    os.makedirs(DRIVE, exist_ok=True)
    if not os.path.exists("/content/image-classification"):
        !git clone -b {BRANCH} {REPO_URL} /content/image-classification
    %cd /content/image-classification
    !git pull -q
    !pip install -q -r requirements.txt
else:
    DRIVE = "data"                                 # локальный запуск из notebooks/
    if os.path.basename(os.getcwd()) == "notebooks":
        %cd ..
    REPO_DIR = os.getcwd()

OUT = f"{DRIVE}/outputs"
ARCHIVE = f"{DRIVE}/oxford_pets_processed.tar"
os.makedirs(OUT, exist_ok=True)

# Данные: при первом запуске скачиваем и уменьшаем (~5 мин), архив кладём на Drive.
# В следующих сессиях просто распаковываем архив (~30 сек) на быстрый локальный диск Colab.
if os.path.exists("data/processed/oxford_pets/train.csv"):
    print("Данные уже подготовлены")
elif os.path.exists(ARCHIVE):
    !mkdir -p data/processed && tar -xf "{ARCHIVE}" -C data/processed
else:
    !python scripts/prepare_data.py
    if IN_COLAB:
        !tar -cf "{ARCHIVE}" -C data/processed oxford_pets
!ls data/processed/oxford_pets

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## TensorBoard
Кривые обновляются во время обучения (кнопка обновления справа сверху).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{OUT}"

## Все эксперименты
12 запусков: 4 архитектуры × 3 режима (`scratch`, `feature_extract`, `finetune`).
На T4 это примерно 1–1.5 часа.

In [ ]:
%cd {REPO_DIR}
!bash scripts/run_all.sh --output-root "{OUT}"

Один эксперимент отдельно (пример):

In [ ]:
# !python scripts/train.py --config configs/experiments/resnet18_finetune.yaml --output-root "{OUT}"

## Сводная таблица и графики

In [ ]:
%cd {REPO_DIR}
!python scripts/make_report.py --output-root "{OUT}"
from IPython.display import Image, Markdown, display
display(Image("results/curves.png"))
display(Image("results/test_accuracy.png"))

## Разбор лучшей модели: отчёт по классам и матрица ошибок

In [ ]:
%cd {REPO_DIR}
import pandas as pd
res = pd.read_csv("results/results.csv")
best = res.loc[res["test_acc"].idxmax(), "experiment"]
print("Лучший эксперимент:", best)
!python scripts/evaluate.py --config configs/experiments/{best}.yaml --output-root "{OUT}"
!cp "{OUT}/{best}/eval_test/confusion_matrix.png" results/confusion_matrix_best.png
!cp "{OUT}/{best}/eval_test/top_confusions.json" results/top_confusions_best.json
display(Image("results/confusion_matrix_best.png"))

## Забрать результаты в репозиторий
Скачайте `results.zip` (копия также сохраняется на Drive), распакуйте в корень локального
репозитория и закоммитьте папку `results/`: там только маленькие csv/md/png, весов нет.

In [ ]:
%cd {REPO_DIR}
!cp -r results "{DRIVE}/"
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")